# 1. Initializations

## 1.1 General imports

In [ ]:
### general
import logging
from smartcheck.logger_config import setup_logger
setup_logger(logging.INFO)
import itertools

### data management
import pandas as pd
import numpy as np

### machine learning (scikit-learn)
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer

### graphical
import matplotlib.pyplot as plt
# for jupyter notebook management
%matplotlib inline

## 1.2 Dataframe imports

In [ ]:
import smartcheck.dataframe_common as dfc
import smartcheck.dataframe_project_specific as dfps

## 1.3 Preprocessing & Modeling imports

In [ ]:
import smartcheck.preprocessing_project_specific as pps
import smartcheck.modeling_project_specific as mps

# 2. Loading and Preprocessing

In [ ]:
df_cpt_raw = dfc.load_dataset_from_config('velo_comptage_ml_ready_data', sep=',', index_col=0)

if df_cpt_raw is not None and isinstance(df_cpt_raw, pd.DataFrame):
    df_cpt = df_cpt_raw.copy()

In [ ]:
df_cpt.info()

## 2.1 Preprocessing pipelines

In [ ]:
keep_cols = [
    "nom_du_site_de_comptage",
    "comptage_horaire",
    "date_et_heure_de_comptage",
    "orientation_compteur",
    "latitude",
    "longitude",
    "arrondissement",
    "jour_ferie",
    "vacances_scolaires",
    "temperature_2m_c",
    "rain_mm",
    "snowfall_cm",
    # "weather_code_wmo_code",
    "elevation",
    "weather_code_wmo_code_category",
]

pipe_preproc = Pipeline([
    ("filter_columns", pps.ColumnFilterTransformer(columns_to_keep=keep_cols)),
    ("add_datetime_features", pps.DatetimePeriodicsTransformer(timestamp_col="date_et_heure_de_comptage")),
])

df_preproc = pipe_preproc.fit_transform(df_cpt)
if df_preproc is not None and isinstance(df_preproc, pd.DataFrame):
    df = df_preproc.copy()

In [ ]:
# Verification des distributions après preprocessing
df.info()
display(df.select_dtypes(include=np.number).describe())
display(df.select_dtypes(include='object').describe())

#### Feature selection (retour d'expérience sur VIF d'autre notebook pour même cible de comparaison)

In [ ]:
col_to_drop1 = [
    'date_et_heure_de_comptage_day_of_year',
    'date_et_heure_de_comptage_year',
    'date_et_heure_de_comptage_month',
    'date_et_heure_de_comptage_day',
    'date_et_heure_de_comptage_day_of_week',
    'date_et_heure_de_comptage_hour'
]

In [ ]:
col_to_drop2 = [
    'latitude',
    'longitude',
    'date_et_heure_de_comptage_sin_month',
    'date_et_heure_de_comptage_sin_week',
]

In [ ]:
col_to_drop3 = [
    'date_et_heure_de_comptage_cos_month',
]

In [ ]:
df = df.drop(columns=col_to_drop1+col_to_drop2+col_to_drop3)
df_feat_sel = df.copy()

## 2.2 Column Transformers

#### Categorical and Numerical

In [ ]:
num_col = list(df.drop(columns='comptage_horaire').select_dtypes(include=np.number).columns)
cat_col = list(df.select_dtypes(include='object').columns)
# s_scaler = StandardScaler()
mm_scaler = MinMaxScaler()
ohe_enc = OneHotEncoder(
    # drop='first', # évites la multicolinéarité mais déclenche des warning (les catégories inconnues participent a renforcer la catégorie droppée...)
    handle_unknown='ignore'
)
tr_num_col = Pipeline(
    steps = [
        ('standardisation', mm_scaler)
    ]
)
tr_cat_col = Pipeline(
    steps = [
        ('encoder', ohe_enc)
    ]
)
tr_columns = ColumnTransformer(
    transformers=[ 
        ('num_col_transf', tr_num_col, num_col),
        ('cat_col_transf', tr_cat_col, cat_col)
    ],
    remainder='drop',
)

# 3. Regression modeling

In [ ]:
liste_compteurs = [
    ('Pont de Bercy', 'NE-SO'),
    ('Pont de Bercy', 'NE-SO'),
    ('135 avenue Daumesnil', 'SE-NO'),
    ("180 avenue d'Italie", 'N-S'),
    ('27 quai de la Tournelle', 'NO-SE'),
    ('27 quai de la Tournelle', 'SE-NO'),
]

## 3.1 Linear Regresion

In [ ]:
pipe_linear_regression = Pipeline(
    steps= [
        ('prep', tr_columns), 
        ('reg',LinearRegression())
    ]
)

#### SANS AR(1) MA(24)

In [ ]:
models_lr_simple_results = {}
# pas d'aggrégation, juste un regroupement par nom de site et orientation (utilisé ensuite dans la boucle for)
grouped = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])
for compteur_id, df_compteur in grouped:
    if compteur_id not in liste_compteurs:
        continue
    # tri chrono + pipeline prétraitement + split
    df_compteur = df_compteur.sort_values("date_et_heure_de_comptage_local")
    X_train, X_train_dates, X_test, X_test_dates, y_train, y_test = \
        dfps.train_test_split_time_aware(
            df_compteur,
            timestamp_cols=["date_et_heure_de_comptage_utc", "date_et_heure_de_comptage_local"],
            target_col="comptage_horaire"
        )
    # pipeline + fit
    pipe_generic = pipe_linear_regression.fit(X_train, y_train)
    # predictions
    y_test_pred = pipe_generic.predict(X_test)
    # stockage des resultats
    models_lr_simple_results[compteur_id] = {
        "pipe":pipe_generic,
        "X_train":X_train,
        "X_train_dates":X_train_dates,
        "X_test":X_test,
        "X_test_dates":X_test_dates,
        "y_train":y_train,
        "y_test":y_test,
        "y_test_pred":y_test_pred,
    }

#### AVEC AR(1) MA(24)

In [ ]:
models_lr_ar1ma24_results = {}

# Grouping by site + orientation
grouped = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])

for compteur_id, df_compteur in grouped:
    if compteur_id not in liste_compteurs:
        continue
    df_compteur = df_compteur.sort_values("date_et_heure_de_comptage_local")

    # Chronological split
    X_train, X_train_dates, X_test, X_test_dates, y_train, y_test = dfps.train_test_split_time_aware(
        df_compteur,
        timestamp_cols=[
            "date_et_heure_de_comptage_utc",
            "date_et_heure_de_comptage_local"
        ],
        target_col="comptage_horaire"
    )
    # Fit AR feature transformer on training set
    tr_ar_feats = pps.AutoregressiveFeaturesTransformer(nb_ar=1, nb_mm=1, roll_wind=24)
    X_train_ar, X_train_dates_ar, y_train_ar = tr_ar_feats.fit_transform(
        X_train, X_train_dates, y_train
    )
    # Train model
    pipe_generic = pipe_linear_regression.fit(X_train_ar, y_train_ar)

    # Transform test set without refitting
    X_test_ar, X_test_dates_ar, y_test_ar = tr_ar_feats.transform_test(
        X_test, X_test_dates, y_test
    )

    # Predict
    y_test_pred = pipe_generic.predict(X_test_ar)

    # Save results
    models_lr_ar1ma24_results[compteur_id] = {
        "pipe":pipe_generic,
        "X_train":X_train_ar,
        "X_train_dates":X_train_dates_ar,
        "X_test":X_test_ar,
        "X_test_dates":X_test_dates_ar,
        "y_train":y_train_ar,
        "y_test":y_test_ar,
        "y_test_pred":y_test_pred,
    }

## 3.1 RandomForest Regresion

In [ ]:
pipe_rfr_regression = Pipeline(
    steps= [
        ('prep', tr_columns), 
        ('reg',RandomForestRegressor(
            random_state=210995,
            n_jobs=-1,
            n_estimators=100,
            max_depth=25,
            min_samples_leaf=5,
            max_features='sqrt'
        ))
    ]
)

#### SANS AR(1) MA(24)

In [ ]:
models_rfr_simple_results = {}
# pas d'aggrégation, juste un regroupement par nom de site et orientation (utilisé ensuite dans la boucle for)
grouped = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])
for compteur_id, df_compteur in grouped:
    if compteur_id not in liste_compteurs:
        continue
    # tri chrono + pipeline prétraitement + split
    df_compteur = df_compteur.sort_values("date_et_heure_de_comptage_local")
    X_train, X_train_dates, X_test, X_test_dates, y_train, y_test = \
        dfps.train_test_split_time_aware(
            df_compteur,
            timestamp_cols=["date_et_heure_de_comptage_utc", "date_et_heure_de_comptage_local"],
            target_col="comptage_horaire"
        )
    # pipeline + fit
    pipe_generic = pipe_rfr_regression.fit(X_train, y_train)
    # predictions
    y_test_pred = pipe_generic.predict(X_test)
    # stockage des resultats
    models_rfr_simple_results[compteur_id] = {
        "pipe":pipe_generic,
        "X_train":X_train,
        "X_train_dates":X_train_dates,
        "X_test":X_test,
        "X_test_dates":X_test_dates,
        "y_train":y_train,
        "y_test":y_test,
        "y_test_pred":y_test_pred,
    }

#### AVEC AR(1) MA(24)

In [ ]:
models_rfr_ar1ma24_results = {}

# Grouping by site + orientation
grouped = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])

for compteur_id, df_compteur in grouped:
    if compteur_id not in liste_compteurs:
        continue
    
    df_compteur = df_compteur.sort_values("date_et_heure_de_comptage_local")

    # Chronological split
    X_train, X_train_dates, X_test, X_test_dates, y_train, y_test = dfps.train_test_split_time_aware(
        df_compteur,
        timestamp_cols=[
            "date_et_heure_de_comptage_utc",
            "date_et_heure_de_comptage_local"
        ],
        target_col="comptage_horaire"
    )
    # Fit AR feature transformer on training set
    tr_ar_feats = pps.AutoregressiveFeaturesTransformer(nb_ar=1, nb_mm=1, roll_wind=24)
    X_train_ar, X_train_dates_ar, y_train_ar = tr_ar_feats.fit_transform(
        X_train, X_train_dates, y_train
    )
    # Train model
    pipe_generic = pipe_rfr_regression.fit(X_train_ar, y_train_ar)

    # Transform test set without refitting
    X_test_ar, X_test_dates_ar, y_test_ar = tr_ar_feats.transform_test(
        X_test, X_test_dates, y_test
    )

    # Predict
    y_test_pred = pipe_generic.predict(X_test_ar)

    # Save results
    models_rfr_ar1ma24_results[compteur_id] = {
        "pipe":pipe_generic,
        "X_train":X_train_ar,
        "X_train_dates":X_train_dates_ar,
        "X_test":X_test_ar,
        "X_test_dates":X_test_dates_ar,
        "y_train":y_train_ar,
        "y_test":y_test_ar,
        "y_test_pred":y_test_pred,
    }

# 4 Performance analysis and visualisation

#### Compteurs "Pont de Bercy", "135 avenue Daumesnil", "180 avenue d'Italie", "27 quai de la Tournelle"

In [ ]:
# Afficher une modélisation de compteur spécifique

liste_model_results = {
    "Regression linéaire simple":models_lr_simple_results,
    "Regression linéaire AR1 MM24":models_lr_ar1ma24_results,
    "Random Forest Regressor simple":models_rfr_simple_results,
    "Random Forest Regressor AR1 MM24":models_rfr_ar1ma24_results,
}
for compteur, (model_name, model_results) in itertools.product(liste_compteurs, liste_model_results.items()):
    print("\n", "*"*80, "\n", f"Modèle {model_name} :","\n", "*"*80, "\n")
    model_metrics = mps.compute_metrics(
        model_results[compteur]["y_test"],
        model_results[compteur]["y_test_pred"]
    )
    print("Metriques du modèle:",model_metrics)

    fig_pred = mps.plot_predictions(
        str(compteur),
        model_results[compteur]["X_test_dates"], 
        model_results[compteur]["y_test"], 
        model_results[compteur]["y_test_pred"], 
        periode_limite=('2025-04-01', '2025-04-16')
    )
    plt.show()

    fig1_res, fig2_res, model_res_coeff = mps.compute_residuals_plot(
        str(compteur),
        model_results[compteur]["X_test_dates"], 
        model_results[compteur]["y_test"], 
        model_results[compteur]["y_test_pred"], 
        periode_limite=('2025-04-01', '2025-04-16')
    )
    print("Pente de la droite de régression des résidus dans le temps (dérive) :", model_res_coeff)
    plt.show()

    fig_interp_list = mps.interpret_model(
        model_results[compteur]
    )
    plt.show()